# Homework 3 — Machine Learning for Classification

Machine Learning Zoomcamp 2026.

Dataset: official `course_lead_scoring_2026.csv`, verified by SHA-256.
Missing categorical values are filled with `NA`; numerical values with 0.0.
Train/validation/test split: 3000/1000/1000, using random_state=42.
Encoders are fitted on training data only. The test set is not used for model selection.

Environment: Python 3.11.17, NumPy 2.3.3, Pandas 2.3.2,
scikit-learn 1.7.2.

## Answers

| Question | Answer |
|---|---|
| Q1 | technology — 1,173 observations |
| Q2 | interaction_count and lead_score — correlation 0.915746 |
| Q3 | lead_source — rounded mutual information 0.03 |
| Q4 | 0.65 — unrounded validation accuracy 0.645 |
| Q5 | number_of_courses_viewed — accuracy difference 0.002 |
| Q6 | C = 0.001 — validation accuracy 0.645 |

Q5 compares only the listed answer options.
Q6 selects the highest rounded accuracy, using the smallest C only in a tie.


In [1]:
!wget -O course_lead_scoring_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-12 09:44:09--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.006s  

2026-10-12 09:44:09 (41.7 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [2]:
import hashlib
from pathlib import Path

actual = hashlib.sha256(
    Path("course_lead_scoring_2026.csv").read_bytes()
).hexdigest()

assert actual == "2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9"
print("Dataset verified: OK")

Dataset verified: OK


In [3]:
import pandas as pd

df = pd.read_csv("course_lead_scoring_2026.csv")

print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

Shape: (5000, 9)

Column types:
lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [4]:
categorical = [
    "lead_source", "industry", "employment_status", "location"
]
numerical = [
    "annual_income", "number_of_courses_viewed",
    "interaction_count", "lead_score"
]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("Missing values after preparation:")
print(df.isna().sum())

Missing values after preparation:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [5]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [6]:
correlations = df[numerical].corr()
correlations

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [7]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train["converted"].to_numpy()
y_val = df_val["converted"].to_numpy()
y_test = df_test["converted"].to_numpy()

df_train = df_train.drop(columns="converted")
df_val = df_val.drop(columns="converted")
df_test = df_test.drop(columns="converted")

print("Train / validation / test:",
      len(df_train), len(df_val), len(df_test))

Train / validation / test: 3000 1000 1000


In [8]:
from sklearn.metrics import mutual_info_score

mi_scores = pd.Series({
    column: round(mutual_info_score(df_train[column], y_train), 2)
    for column in categorical
})

mi_scores.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [9]:
from sklearn.feature_extraction import DictVectorizer

features = categorical + numerical

train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print("Encoded training shape:", X_train.shape)
print("Encoded validation shape:", X_val.shape)

Encoded training shape: (3000, 28)
Encoded validation shape: (1000, 28)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
baseline_accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", baseline_accuracy)
print("Rounded Q4 value:", round(baseline_accuracy, 2))

Validation accuracy: 0.645
Rounded Q4 value: 0.65


In [11]:
results = []

for excluded in features:
    remaining = [f for f in features if f != excluded]

    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(
        df_train[remaining].to_dict(orient="records")
    )
    X_val_reduced = dv_reduced.transform(
        df_val[remaining].to_dict(orient="records")
    )

    model_reduced = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    model_reduced.fit(X_train_reduced, y_train)

    accuracy_without = accuracy_score(
        y_val, model_reduced.predict(X_val_reduced)
    )

    results.append({
        "excluded_feature": excluded,
        "accuracy_without": accuracy_without,
        "difference": baseline_accuracy - accuracy_without
    })

elimination_results = pd.DataFrame(results).set_index("excluded_feature")
elimination_results

,accuracy_without,difference
excluded_feature,,
lead_source,0.642,0.003
industry,0.645,0.000
employment_status,0.645,0.000
location,0.645,0.000
annual_income,0.724,-0.079
number_of_courses_viewed,0.643,0.002
interaction_count,0.601,0.044
lead_score,0.644,0.001


In [12]:
regularization_results = []

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model_regularized = LogisticRegression(
        solver="liblinear",
        C=c,
        max_iter=1000,
        random_state=42
    )
    model_regularized.fit(X_train, y_train)

    accuracy = accuracy_score(
        y_val, model_regularized.predict(X_val)
    )

    regularization_results.append({
        "C": c,
        "accuracy": accuracy,
        "rounded_accuracy": round(accuracy, 3)
    })

pd.DataFrame(regularization_results)

,C,accuracy,rounded_accuracy
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645
